In [ ]:
# --- Cell 0.1: Залежності ---
!pip install -q "sentence-transformers>=5.0" "transformers>=4.51.0" qdrant-client pandas numpy scikit-learn accelerate peft

# --- Cell 0.2: Імпорти ---
import os
import glob
import gc
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

from sentence_transformers import CrossEncoder
from sentence_transformers.cross_encoder import (
    CrossEncoderTrainer,
    CrossEncoderTrainingArguments,
)
from sentence_transformers.cross_encoder.losses import (
    CachedMultipleNegativesRankingLoss,
    BinaryCrossEntropyLoss,
)
from datasets import Dataset

from qdrant_client import QdrantClient
from google.colab import drive, userdata

drive.mount("/content/drive")

device_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
print(f"GPU: {device_name}")
USE_BF16 = any(x in device_name for x in ["L4", "A100", "H100", "L40", "A10"])
print(f"bf16: {USE_BF16}  (fp16 НЕ використовуємо — Qwen схильний до NaN)")

# --- Cell 0.3: Конфіг ---
CFG = {
    # Модель
    "student_model": "tomaarsen/Qwen3-Reranker-0.6B-seq-cls",

    # Дані — ОНОВИ ШЛЯХИ під свої
    "train_pairs": "/content/drive/MyDrive/CV_rank_Datasets/train_pairs.csv",
    "auto_eval_pool": "/content/drive/MyDrive/CV_rank_Datasets/auto_eval_pool.csv",
    "golden_eval_dir": "/content/drive/MyDrive/CV_rank_Datasets/Golden_Eval",

    # Qdrant (тільки щоб дістати тексти вакансій по job_id)
    "collection_jobs": "trained_embedder_jobs",

    # Вихід
    "output_dir": "/content/drive/MyDrive/qwen3-reranker-cv-finetuned",

    # Тренування — Qwen 0.6B decoder важчий за bge, батчі менші
    "epochs": 3,
    "batch_size": 4,
    "grad_accum": 4,            # ефективний batch = 16
    "lr": 1e-5,                 # нижчий ніж для bge — Qwen чутливіший
    "max_length": 512,

    # Дистиляція
    "use_distillation": True,
    "distill_epochs": 1,
    "distill_lr": 5e-6,

    "seed": 42,
    "eval_k": 10,
}
print("Config готовий")

# --- Cell 0.4: Qdrant (для текстів вакансій) ---
client = QdrantClient(
    url=userdata.get("QDRANT_URL"),
    api_key=userdata.get("QDRANT_API_KEY"),
    timeout=300,
)
print("Колекції:", [c.name for c in client.get_collections().collections])


# ============================================================================
#  SECTION 1 — TRAIN ДАНІ
# ============================================================================

# --- Cell 1.1: Helpers ---
def strip_instruct(text):
    """Прибирає Qwen-обгортку 'Instruct: ...\\nQuery: <текст>'."""
    text = str(text)
    if "Query:" in text:
        return text.split("Query:", 1)[1].strip()
    return text.strip()


def clip(text, n=2000):
    return str(text)[:n]


# --- Cell 1.2: Завантаження train_pairs ---
train_pairs = pd.read_csv(CFG["train_pairs"])
train_pairs["vacancy_text"] = train_pairs["vacancy_text"].apply(strip_instruct).apply(clip)
train_pairs["cv_text_positive"] = train_pairs["cv_text_positive"].apply(clip)
train_pairs["cv_text_hard_negative"] = train_pairs["cv_text_hard_negative"].apply(clip)

print(f"Train триплетів: {len(train_pairs)}")
print(f"\nПриклад vacancy_text:\n{train_pairs['vacancy_text'].iloc[0][:200]}")

train_dataset = Dataset.from_dict({
    "query": train_pairs["vacancy_text"].tolist(),
    "positive": train_pairs["cv_text_positive"].tolist(),
    "negative": train_pairs["cv_text_hard_negative"].tolist(),
})
print(f"Train dataset: {len(train_dataset)}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GPU: NVIDIA L4
bf16: True  (fp16 НЕ використовуємо — Qwen схильний до NaN)
Config готовий
Колекції: ['candidates', 'jobs', 'trained_embedder_cvs', 'trained_embedder_jobs']
Train триплетів: 1209

Приклад vacancy_text:
Title: Інженер з якості (Quality Engineer)
Company: Eleven
Category: 
Skills: 
Languages: 
Experience: 3 years experience
Employment: FULL_TIME
Location: 
Domain: Other

Description:
Eleven — рекрутин
Train dataset: 1209


In [ ]:
# ============================================================================
#  SECTION 2 — GOLDEN_EVAL (окремий сет)
# ============================================================================
# Тепер це окремий набір — розділяти по вакансіях не треба.
# Модель тренується на train_pairs, оцінюється на Golden_eval. Чисто.

# --- Cell 2.1: Кеш текстів вакансій з Qdrant ---
_vacancy_cache = {}

def get_vacancy_text(job_id):
    if job_id in _vacancy_cache:
        return _vacancy_cache[job_id]
    try:
        points = client.retrieve(
            collection_name=CFG["collection_jobs"],
            ids=[job_id],
            with_payload=True,
        )
        if points:
            p = points[0].payload
            text = strip_instruct(p.get("embed_text", "")) if p.get("embed_text") else ""
            if not text:
                text = " ".join(str(p.get(k, "")) for k in
                                ["title", "skills_tags", "experience_text"])
            _vacancy_cache[job_id] = clip(text)
            return _vacancy_cache[job_id]
    except Exception as e:
        print(f"  ⚠ {job_id}: {e}")
    _vacancy_cache[job_id] = ""
    return ""


# --- Cell 2.2: Збірка golden_eval ---
# Патерн ловить і 'eval_pool_X.csv', і 'Copy of eval_pool_X.csv'
golden_files = sorted(glob.glob(f"{CFG['golden_eval_dir']}/*eval_pool_*.csv"))
if not golden_files:
    # запасний варіант — будь-які csv у папці
    golden_files = sorted(glob.glob(f"{CFG['golden_eval_dir']}/*.csv"))
print(f"Знайдено файлів: {len(golden_files)}")

golden_eval = {}
for path in tqdm(golden_files, desc="Читання Golden_eval"):
    df = pd.read_csv(path)
    if "job_id" not in df.columns or "cv_text" not in df.columns:
        print(f"  ⚠ пропущено {os.path.basename(path)}")
        continue

    job_id = df["job_id"].iloc[0]
    vtext = get_vacancy_text(job_id)
    if not vtext:
        print(f"  ⚠ немає тексту вакансії: {os.path.basename(path)}")
        continue

    cands = [
        (clip(r["cv_text"]), int(r["Relevance_Score_0_to_3"]))
        for _, r in df.iterrows()
        if pd.notna(r.get("cv_text"))
    ]
    if cands:
        golden_eval[job_id] = {
            "title": os.path.basename(path)
                .replace("Copy of ", "").replace("eval_pool_", "").replace("_Final.csv", ""),
            "vacancy_text": vtext,
            "candidates": cands,
        }

print(f"\nGolden_eval вакансій: {len(golden_eval)}")
print(f"Всього кандидатів: {sum(len(v['candidates']) for v in golden_eval.values())}")

# Розподіл labels — важливо для розуміння метрик
from collections import Counter
all_labels = [c[1] for v in golden_eval.values() for c in v["candidates"]]
print(f"Розподіл labels: {dict(sorted(Counter(all_labels).items()))}")

Знайдено файлів: 10


Читання Golden_eval:   0%|          | 0/10 [00:00<?, ?it/s]


Golden_eval вакансій: 10
Всього кандидатів: 629
Розподіл labels: {0: 257, 1: 283, 2: 73, 3: 16}


In [ ]:
# ============================================================================
#  SECTION 3 — МЕТРИКИ
# ============================================================================
# NDCG@k, Recall@k, MAP@k, MRR@k, Precision@k — щоб збігалося з форматом
# таблиці порівняння dense-моделей.

# --- Cell 3.1: Реалізація метрик ---
from sklearn.metrics import ndcg_score


def _relevant_mask(labels, threshold=2):
    """Релевантний = label >= threshold. За замовчуванням 2 і 3."""
    return np.asarray(labels) >= threshold


def compute_metrics(labels, scores, k=10, rel_threshold=2):
    """
    labels — ground truth 0-3
    scores — передбачені скори моделі
    Повертає dict метрик @k.
    """
    labels = np.asarray(labels, dtype=float)
    scores = np.asarray(scores, dtype=float)

    order = np.argsort(-scores)          # спадання
    top_k = order[:k]

    rel = _relevant_mask(labels, rel_threshold)
    n_rel_total = rel.sum()

    # NDCG@k — по градуйованих labels (0-3)
    ndcg = ndcg_score([labels], [scores], k=k) if labels.max() > 0 else 0.0

    # Precision@k / Recall@k — по бінарній релевантності
    hits_at_k = rel[top_k].sum()
    precision = hits_at_k / k
    recall = hits_at_k / n_rel_total if n_rel_total > 0 else 0.0

    # MAP@k — average precision серед top-k
    ap, hit_count = 0.0, 0
    for rank, idx in enumerate(top_k, 1):
        if rel[idx]:
            hit_count += 1
            ap += hit_count / rank
    denom = min(n_rel_total, k)
    map_at_k = ap / denom if denom > 0 else 0.0

    # MRR@k — 1 / позиція першого релевантного
    mrr = 0.0
    for rank, idx in enumerate(top_k, 1):
        if rel[idx]:
            mrr = 1.0 / rank
            break

    return {
        "ndcg": ndcg,
        "precision": precision,
        "recall": recall,
        "map": map_at_k,
        "mrr": mrr,
    }


# --- Cell 3.2: Evaluation реранкера на golden_eval ---
def evaluate_reranker(model, golden, k=10, batch_size=16, rel_threshold=2, desc="Eval"):
    """Прогоняє реранкер по всіх вакансіях golden_eval, усереднює метрики."""
    acc = {"ndcg": [], "precision": [], "recall": [], "map": [], "mrr": []}

    for job_id, data in tqdm(golden.items(), desc=desc, leave=False):
        vacancy = data["vacancy_text"]
        cvs = [c[0] for c in data["candidates"]]
        labels = [c[1] for c in data["candidates"]]

        if max(labels) == 0:
            continue

        pairs = [[vacancy, cv] for cv in cvs]
        scores = model.predict(pairs, batch_size=batch_size, show_progress_bar=False)

        m = compute_metrics(labels, scores, k=k, rel_threshold=rel_threshold)
        for key in acc:
            acc[key].append(m[key])

    return {
        f"NDCG@{k}": float(np.mean(acc["ndcg"])) if acc["ndcg"] else 0.0,
        f"Precision@{k}": float(np.mean(acc["precision"])) if acc["precision"] else 0.0,
        f"Recall@{k}": float(np.mean(acc["recall"])) if acc["recall"] else 0.0,
        f"MAP@{k}": float(np.mean(acc["map"])) if acc["map"] else 0.0,
        f"MRR@{k}": float(np.mean(acc["mrr"])) if acc["mrr"] else 0.0,
        "n_vacancies": len(acc["ndcg"]),
    }


def print_metrics(name, m):
    print(f"\n=== {name} ===")
    for key, val in m.items():
        if key == "n_vacancies":
            print(f"  {'вакансій':<14} {val}")
        else:
            print(f"  {key:<14} {val:.4f}")

In [ ]:
# ============================================================================
#  SECTION 4 — BASELINE (до тренування)
# ============================================================================

# --- Cell 4.1: Baseline вимір ---
print(f"Завантаження baseline: {CFG['student_model']}")
baseline_model = CrossEncoder(
    CFG["student_model"],
    max_length=CFG["max_length"],
    model_kwargs={"torch_dtype": torch.bfloat16} if USE_BF16 else {},
)

baseline_metrics = evaluate_reranker(
    baseline_model, golden_eval, k=CFG["eval_k"], desc="Baseline"
)
print_metrics("BASELINE (Qwen3-Reranker, без тренування)", baseline_metrics)

# Звільняємо пам'ять перед тренуванням
del baseline_model
gc.collect()
torch.cuda.empty_cache()

Завантаження baseline: tomaarsen/Qwen3-Reranker-0.6B-seq-cls


config.json:   0%|          | 0.00/844 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.38GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

...

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

Baseline:   0%|          | 0/10 [00:00<?, ?it/s]


=== BASELINE (Qwen3-Reranker, без тренування) ===
  NDCG@10        0.3244
  Precision@10   0.1500
  Recall@10      0.1644
  MAP@10         0.0808
  MRR@10         0.3560
  вакансій       10


In [ ]:
# ============================================================================
#  SECTION 5 — STAGE 1: PAIRWISE FINE-TUNING
# ============================================================================

# --- Cell 5.1: Модель для тренування ---
# ВАЖЛИВО: для тренування вантажимо у fp32 і покладаємось на bf16 автокаст
# у Trainer. Завантаження одразу в half + тренування = та сама NaN-проблема,
# яку ми ловили на Qwen3-Embedding.
model = CrossEncoder(
    CFG["student_model"],
    max_length=CFG["max_length"],
    num_labels=1,
)
print(f"Student: {CFG['student_model']}")
print(f"dtype: {next(model.model.parameters()).dtype}  (має бути float32)")

# --- Cell 5.2: Loss ---
loss = CachedMultipleNegativesRankingLoss(model, mini_batch_size=2)
print("Loss: CachedMultipleNegativesRankingLoss (pairwise)")

# --- Cell 5.3: Training arguments ---
args = CrossEncoderTrainingArguments(
    output_dir=CFG["output_dir"],
    num_train_epochs=CFG["epochs"],
    per_device_train_batch_size=CFG["batch_size"],
    gradient_accumulation_steps=CFG["grad_accum"],
    learning_rate=CFG["lr"],
    warmup_ratio=0.1,

    # Точність: тільки bf16, ніколи fp16 на Qwen
    fp16=False,
    bf16=USE_BF16,

    # Стабільність
    max_grad_norm=1.0,
    gradient_checkpointing=False,   # конфліктує з Cached loss
    dataloader_num_workers=0,

    eval_strategy="no",             # eval робимо вручну після кожної стадії
    save_strategy="epoch",
    save_total_limit=2,
    logging_steps=10,               # часте логування — щоб зловити NaN рано
    seed=CFG["seed"],
)

# --- Cell 5.4: Тренування ---
trainer = CrossEncoderTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    loss=loss,
)

print("Stage 1: pairwise fine-tuning...")
print("СТЕЖ ЗА LOSS: якщо стане nan — зупиняй одразу, знижуй lr до 5e-6")
trainer.train()

model.save_pretrained(CFG["output_dir"])
print(f"Stage 1 збережено → {CFG['output_dir']}")

# --- Cell 5.5: Метрики після Stage 1 ---
stage1_metrics = evaluate_reranker(
    model, golden_eval, k=CFG["eval_k"], desc="Stage 1"
)
print_metrics("ПІСЛЯ STAGE 1 (pairwise)", stage1_metrics)

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Student: tomaarsen/Qwen3-Reranker-0.6B-seq-cls
dtype: torch.float32  (має бути float32)
Loss: CachedMultipleNegativesRankingLoss (pairwise)


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Stage 1: pairwise fine-tuning...
СТЕЖ ЗА LOSS: якщо стане nan — зупиняй одразу, знижуй lr до 5e-6


Step,Training Loss
10,4.208738
20,2.383573
30,1.472040
40,1.159950
50,1.449632
60,1.010402
70,0.855375
80,0.766249
90,0.463203
100,0.371200


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Stage 1 збережено → /content/drive/MyDrive/qwen3-reranker-cv-finetuned


Stage 1:   0%|          | 0/10 [00:00<?, ?it/s]


=== ПІСЛЯ STAGE 1 (pairwise) ===
  NDCG@10        0.5649
  Precision@10   0.2800
  Recall@10      0.2349
  MAP@10         0.2056
  MRR@10         0.6333
  вакансій       10


In [ ]:
# ============================================================================
#  SECTION 6 — STAGE 2: DISTILLATION (опційно)
# ============================================================================
# Донавчання на soft scores від Qwen3-Reranker-teacher.
# Легкий калібрувальний прохід: 1 епоха, дуже низький lr.

# --- Cell 6.1: Очистка пам'яті ---
del trainer, loss
gc.collect()
torch.cuda.empty_cache()
print(f"Free VRAM: "
      f"{(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1e9:.2f} GB")

# --- Cell 6.2: Distillation дані ---
if CFG["use_distillation"]:
    auto_df = pd.read_csv(CFG["auto_eval_pool"])
    auto_df["cv_text"] = auto_df["cv_text"].apply(clip)
    print(f"Distillation пар: {len(auto_df)}")
    print(f"auto_score: mean={auto_df['auto_score'].mean():.3f}, "
          f"min={auto_df['auto_score'].min():.3f}, max={auto_df['auto_score'].max():.3f}")

    distill_dataset = Dataset.from_dict({
        "query": auto_df["vacancy"].astype(str).tolist(),
        "doc": auto_df["cv_text"].tolist(),
        "label": auto_df["auto_score"].astype(float).tolist(),
    })

    # --- Cell 6.3: Distillation тренування ---
    distill_loss = BinaryCrossEntropyLoss(model)

    distill_args = CrossEncoderTrainingArguments(
        output_dir=CFG["output_dir"] + "-distilled",
        num_train_epochs=CFG["distill_epochs"],
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        learning_rate=CFG["distill_lr"],
        warmup_ratio=0.1,
        fp16=False,
        bf16=USE_BF16,
        max_grad_norm=1.0,
        gradient_checkpointing=False,
        dataloader_num_workers=0,
        eval_strategy="no",
        save_strategy="epoch",
        save_total_limit=1,
        logging_steps=10,
        seed=CFG["seed"],
    )

    distill_trainer = CrossEncoderTrainer(
        model=model,
        args=distill_args,
        train_dataset=distill_dataset,
        loss=distill_loss,
    )

    print("Stage 2: distillation...")
    distill_trainer.train()

    model.save_pretrained(CFG["output_dir"] + "-distilled")
    print(f"Stage 2 збережено → {CFG['output_dir']}-distilled")

    # --- Cell 6.4: Метрики після Stage 2 ---
    stage2_metrics = evaluate_reranker(
        model, golden_eval, k=CFG["eval_k"], desc="Stage 2"
    )
    print_metrics("ПІСЛЯ STAGE 2 (+ distillation)", stage2_metrics)
else:
    stage2_metrics = None

Free VRAM: 21.26 GB


Distillation пар: 1134
auto_score: mean=0.253, min=0.000, max=0.992
Stage 2: distillation...


Step,Training Loss
10,0.812145
20,0.577914
30,0.506385
40,0.546563
50,0.569117
60,0.477614
70,0.531054


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Stage 2 збережено → /content/drive/MyDrive/qwen3-reranker-cv-finetuned-distilled


Stage 2:   0%|          | 0/10 [00:00<?, ?it/s]


=== ПІСЛЯ STAGE 2 (+ distillation) ===
  NDCG@10        0.5181
  Precision@10   0.2400
  Recall@10      0.2084
  MAP@10         0.1518
  MRR@10         0.5000
  вакансій       10


In [ ]:
# --- Cell 7.1: Зведена таблиця ---
rows = [
    {"Модель": "Qwen3-Reranker (baseline)", **baseline_metrics},
    {"Модель": "+ pairwise fine-tune", **stage1_metrics},
]
if stage2_metrics:
    rows.append({"Модель": "+ distillation", **stage2_metrics})

results_df = pd.DataFrame(rows).drop(columns=["n_vacancies"])
k = CFG["eval_k"]

print("\n" + "=" * 80)
print(f"РЕЗУЛЬТАТИ на Golden_eval ({baseline_metrics['n_vacancies']} вакансій)")
print("=" * 80)
print(results_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

# Дельти від baseline
print(f"\nΔ від baseline:")
for col in [f"NDCG@{k}", f"Recall@{k}", f"MAP@{k}", f"MRR@{k}"]:
    base = results_df[col].iloc[0]
    best = results_df[col].iloc[-1]
    rel = (best - base) / base * 100 if base > 0 else 0
    print(f"  {col:<14} {base:.4f} → {best:.4f}  ({best - base:+.4f}, {rel:+.1f}%)")

# --- Cell 7.2: Збереження результатів ---
results_df.to_csv("/content/drive/MyDrive/qwen_reranker_results.csv", index=False)
print("\nЗбережено → /content/drive/MyDrive/qwen_reranker_results.csv")


РЕЗУЛЬТАТИ на Golden_eval (10 вакансій)
                   Модель  NDCG@10  Precision@10  Recall@10  MAP@10  MRR@10
Qwen3-Reranker (baseline)   0.3244        0.1500     0.1644  0.0808  0.3560
     + pairwise fine-tune   0.5649        0.2800     0.2349  0.2056  0.6333
           + distillation   0.5181        0.2400     0.2084  0.1518  0.5000

Δ від baseline:
  NDCG@10        0.3244 → 0.5181  (+0.1937, +59.7%)
  Recall@10      0.1644 → 0.2084  (+0.0440, +26.8%)
  MAP@10         0.0808 → 0.1518  (+0.0710, +87.8%)
  MRR@10         0.3560 → 0.5000  (+0.1440, +40.5%)

Збережено → /content/drive/MyDrive/qwen_reranker_results.csv


In [ ]:
# ============================================================================
#  ДЕПЛОЙ
# ============================================================================
#
#  from sentence_transformers import CrossEncoder
#  reranker = CrossEncoder("/content/drive/MyDrive/qwen3-reranker-cv-finetuned-distilled")
#
#  pairs = [[vacancy_text, cv_text] for cv_text in candidate_cvs]
#  scores = reranker.predict(pairs)
#
#  Для фінального порівняння (pretrain vs FT embedding vs embedding+rerank)
#  — окремий ноутбук, там знадобиться ivvvvvi/cv_embedder.
# ============================================================================

In [ ]:
!pip install -q huggingface_hub

from huggingface_hub import HfApi, login
from google.colab import userdata

login(token=userdata.get("HF_TOKEN"))

REPO_ID = "Driulik/cv_reranker_qwen"   # ← зміни на свій username/назву
MODEL_PATH = "/content/drive/MyDrive/qwen3-reranker-cv-finetuned"  # pairwise, не distilled

api = HfApi()
api.create_repo(repo_id=REPO_ID, exist_ok=True, private=False)

api.upload_folder(
    folder_path=MODEL_PATH,
    repo_id=REPO_ID,
    commit_message="Fine-tuned Qwen3-Reranker-0.6B for CV-vacancy ranking",
    ignore_patterns=["checkpoint-*", "*.pt", "optimizer.pt", "scheduler.pt"],
)

print(f"✅ https://huggingface.co/{REPO_ID}")

✅ https://huggingface.co/Driulik/cv_reranker_qwen
